### Basic Analysis

In [ ]:
import pandas as pd

Importing Dataset

In [ ]:
df=pd.read_csv("/content/sample_data/api_data_aadhar_enrolment_0_500000.csv")
df.head(5)

,date,state,district,pincode,age_0_5,age_5_17,age_18_greater
0,02-03-2025,Meghalaya,East Khasi Hills,793121,11,61,37.0
1,09-03-2025,Karnataka,Bengaluru Urban,560043,14,33,39.0
2,09-03-2025,Uttar Pradesh,Kanpur Nagar,208001,29,82,12.0
3,09-03-2025,Uttar Pradesh,Aligarh,202133,62,29,15.0
4,09-03-2025,Karnataka,Bengaluru Urban,560016,14,16,21.0


Finding Missing Values

In [ ]:
df.isna().sum()

,0
date,0
state,0
district,0
pincode,0
age_0_5,0
age_5_17,0
age_18_greater,1


In [ ]:
df.isnull().sum()

,0
date,0
state,0
district,0
pincode,0
age_0_5,0
age_5_17,0
age_18_greater,1


Converting to DateTime format

In [ ]:
import pandas as pd
df['date'] = pd.to_datetime(df['date'], dayfirst=True)
df['day'] = df['date'].dt.day
df['month'] = df['date'].dt.month
df['quarter'] = df['date'].dt.quarter
df.head()

,date,state,district,pincode,age_0_5,age_5_17,age_18_greater,day,month,quarter
0,2025-03-02,Meghalaya,East Khasi Hills,793121,11,61,37.0,2,3,1
1,2025-03-09,Karnataka,Bengaluru Urban,560043,14,33,39.0,9,3,1
2,2025-03-09,Uttar Pradesh,Kanpur Nagar,208001,29,82,12.0,9,3,1
3,2025-03-09,Uttar Pradesh,Aligarh,202133,62,29,15.0,9,3,1
4,2025-03-09,Karnataka,Bengaluru Urban,560016,14,16,21.0,9,3,1


Loading Secondary Dataset

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("sirpunch/indian-census-data-with-geospatial-indexing")

print("Path to dataset files:", path)

100%|██████████| 43.4k/43.4k [00:00<00:00, 2.12MB/s]

Extracting files...
Path to dataset files: /root/.cache/kagglehub/datasets/sirpunch/indian-census-data-with-geospatial-indexing/versions/1


In [ ]:
df_2 = pd.read_csv("/root/.cache/kagglehub/datasets/sirpunch/indian-census-data-with-geospatial-indexing/versions/1/district wise population and centroids.csv")
df_2.head()

,State,District,Latitude,Longitude,Population in 2001,Population in 2011
0,Andhra Pradesh,Anantapur,14.312066,77.460158,3640478,4081148
1,Andhra Pradesh,Chittoor,13.331093,78.927639,3745875,4174064
2,Andhra Pradesh,East Godavari,16.782718,82.243207,4901420,5154296
3,Andhra Pradesh,Guntur,15.884926,80.586576,4465144,4887813
4,Andhra Pradesh,Krishna,16.143873,81.148051,4187841,4517398


Exploratory Data Analysis of Secondary Dataset

In [ ]:
df_2.columns

Index(['State', 'District', 'Latitude', 'Longitude', 'Population in 2001',
       'Population in 2011'],
      dtype='object')

In [ ]:
df_2 = df_2.rename(columns={"State": "state", "District": "district"})
df_2 = df_2.drop(columns=["Population in 2001", "Population in 2011"], errors='ignore')
df_2.head()

,state,district,Latitude,Longitude
0,Andhra Pradesh,Anantapur,14.312066,77.460158
1,Andhra Pradesh,Chittoor,13.331093,78.927639
2,Andhra Pradesh,East Godavari,16.782718,82.243207
3,Andhra Pradesh,Guntur,15.884926,80.586576
4,Andhra Pradesh,Krishna,16.143873,81.148051


Merging both datasets

In [ ]:
df_main = df_2.merge(
    df,
    on=["state", "district"],
    how="left"
)
df_main.head()

,state,district,Latitude,Longitude,date,pincode,age_0_5,age_5_17,age_18_greater,day,month,quarter,year_month
0,Andhra Pradesh,Anantapur,14.312066,77.460158,2025-09-01,515672.0,2.0,0.0,0.0,1.0,9.0,3.0,2025-09
1,Andhra Pradesh,Anantapur,14.312066,77.460158,2025-09-01,515761.0,1.0,0.0,0.0,1.0,9.0,3.0,2025-09
2,Andhra Pradesh,Anantapur,14.312066,77.460158,2025-09-01,515842.0,3.0,0.0,0.0,1.0,9.0,3.0,2025-09
3,Andhra Pradesh,Anantapur,14.312066,77.460158,2025-09-01,515867.0,1.0,0.0,0.0,1.0,9.0,3.0,2025-09
4,Andhra Pradesh,Anantapur,14.312066,77.460158,2025-09-01,515872.0,1.0,0.0,0.0,1.0,9.0,3.0,2025-09


Checking for missing values in main dataset

In [ ]:
df_main.isna().sum()

,0
state,0
district,0
Latitude,0
Longitude,0
date,0
pincode,0
age_0_5,0
age_5_17,0
age_18_greater,0
day,0


## Impute missing values using KNN Imputer



In [ ]:
from sklearn.impute import KNNImputer
print("KNNImputer imported successfully.")

KNNImputer imported successfully.


Imputing the columns with potential missing values

In [ ]:
imputer = KNNImputer()

columns_to_impute_knn = ['pincode', 'age_0_5', 'age_5_17', 'age_18_greater']

# Apply KNNImputer
df_main[columns_to_impute_knn] = imputer.fit_transform(df_main[columns_to_impute_knn])

# Impute missing values in 'date' column with the mode
if df_main['date'].isnull().any():
    mode_date = df_main['date'].mode()[0]
    df_main['date'].fillna(mode_date, inplace=True)

df_main['date'] = pd.to_datetime(df_main['date'], dayfirst=True)

df_main['day'] = df_main['date'].dt.day
df_main['month'] = df_main['date'].dt.month
df_main['quarter'] = df_main['date'].dt.quarter
df_main['year_month'] = df_main['date'].dt.to_period('M').astype(str)

print("Missing values after imputation and reconstruction:")
print(df_main[['pincode', 'age_0_5', 'age_5_17', 'age_18_greater', 'date', 'day', 'month', 'quarter', 'year_month']].isnull().sum())

Missing values after imputation and reconstruction:
pincode           0
age_0_5           0
age_5_17          0
age_18_greater    0
date              0
day               0
month             0
quarter           0
year_month        0
dtype: int64


/tmp/ipython-input-609562842.py:12: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df_main['date'].fillna(mode_date, inplace=True)


### Geographical Classification

In [ ]:
import numpy as np


# Latitude thresholds
north_latitude_threshold = 28
south_latitude_threshold = 16

# Longitude thresholds
west_longitude_threshold = 75
east_longitude_threshold = 85

conditions = [
    (df_main['Latitude'] > north_latitude_threshold), # North
    (df_main['Latitude'] < south_latitude_threshold), # South
    (df_main['Longitude'] > east_longitude_threshold), # East
    (df_main['Longitude'] < west_longitude_threshold)  # West
]

choices = ['North', 'South', 'East', 'West']
df_main['geographical_zone'] = np.select(conditions, choices, default='Central')

print("District geographical zones classified successfully.")
print("\nDistribution of geographical zones:")
print(df_main['geographical_zone'].value_counts())


District geographical zones classified successfully.

Distribution of geographical zones:
geographical_zone
Central    4834
East       2837
South      2685
West       1932
North      1076
Name: count, dtype: int64


Loading Geomapping.csv containing zones nationwide for different states

In [ ]:
import pandas as pd
geomapping_file_path = "/geomapping.csv"

try:
    df_geomap = pd.read_csv(geomapping_file_path)
    print(f"Successfully loaded {geomapping_file_path}")
    df_geomap.rename(columns={'State / UT': 'state'}, inplace=True)

    # Perform the merge. Using a left merge to keep all records from df_main.
    df_main = df_main.merge(df_geomap, on='state', how='left')

    print("\nMerged topographical data with df_main. Displaying head of the updated df_main:")
    display(df_main.head())

    print("\nChecking for missing values in newly merged columns (e.g., elevation, if present):")
    if 'Elevation' in df_geomap.columns:
        print(df_main['Elevation'].isnull().sum())
    elif 'elevation' in df_geomap.columns:
        print(df_main['elevation'].isnull().sum())
    else:
        print("No 'Elevation' or 'elevation' column found in geomapping data to check for nulls.")

except FileNotFoundError:
    print(f"Error: The file '{geomapping_file_path}' was not found. Please provide the correct path.")
except Exception as e:
    print(f"An error occurred during file loading or merging: {e}")

Successfully loaded /geomapping.csv

Merged topographical data with df_main. Displaying head of the updated df_main:


,state,district,Latitude,Longitude,date,pincode,age_0_5,age_5_17,age_18_greater,day,month,quarter,year_month,geographical_zone,Topographical Region
0,Andhra Pradesh,Anantapur,14.312066,77.460158,2025-09-01,515672.0,2.0,0.0,0.0,1,9,3,2025-09,South,Coastal
1,Andhra Pradesh,Anantapur,14.312066,77.460158,2025-09-01,515761.0,1.0,0.0,0.0,1,9,3,2025-09,South,Coastal
2,Andhra Pradesh,Anantapur,14.312066,77.460158,2025-09-01,515842.0,3.0,0.0,0.0,1,9,3,2025-09,South,Coastal
3,Andhra Pradesh,Anantapur,14.312066,77.460158,2025-09-01,515867.0,1.0,0.0,0.0,1,9,3,2025-09,South,Coastal
4,Andhra Pradesh,Anantapur,14.312066,77.460158,2025-09-01,515872.0,1.0,0.0,0.0,1,9,3,2025-09,South,Coastal



Checking for missing values in newly merged columns (e.g., elevation, if present):
No 'Elevation' or 'elevation' column found in geomapping data to check for nulls.


In [ ]:
state_centers = (
    df_main.groupby("state")[["Latitude", "Longitude"]]
    .mean()
    .reset_index()
)


In [ ]:
def assign_geo_zone(row, centers):
    sc = centers[centers["state"] == row["state"]].iloc[0]

    lat_diff = row["Latitude"] - sc["Latitude"]
    lon_diff = row["Longitude"] - sc["Longitude"]

    if abs(lat_diff) < 0.25 and abs(lon_diff) < 0.25:
        return "Central"
    elif lat_diff >= 0 and abs(lat_diff) >= abs(lon_diff):
        return "North"
    elif lat_diff < 0 and abs(lat_diff) >= abs(lon_diff):
        return "South"
    elif lon_diff >= 0:
        return "East"
    else:
        return "West"


In [ ]:
df_main["Geographical_Zone"] = df_main.apply(assign_geo_zone, axis=1, centers=state_centers)
df_main.head()

,state,district,Latitude,Longitude,date,pincode,age_0_5,age_5_17,age_18_greater,day,month,quarter,year_month,geographical_zone,Topographical Region,Geographical_Zone
0,Andhra Pradesh,Anantapur,14.312066,77.460158,2025-09-01,515672.0,2.0,0.0,0.0,1,9,3,2025-09,South,Coastal,West
1,Andhra Pradesh,Anantapur,14.312066,77.460158,2025-09-01,515761.0,1.0,0.0,0.0,1,9,3,2025-09,South,Coastal,West
2,Andhra Pradesh,Anantapur,14.312066,77.460158,2025-09-01,515842.0,3.0,0.0,0.0,1,9,3,2025-09,South,Coastal,West
3,Andhra Pradesh,Anantapur,14.312066,77.460158,2025-09-01,515867.0,1.0,0.0,0.0,1,9,3,2025-09,South,Coastal,West
4,Andhra Pradesh,Anantapur,14.312066,77.460158,2025-09-01,515872.0,1.0,0.0,0.0,1,9,3,2025-09,South,Coastal,West


Aggregation of same districts per date

In [ ]:
agg_funcs = {
    'age_0_5': 'sum',
    'age_5_17': 'sum',
    'age_18_greater': 'sum',
    'Latitude': 'first',
    'Longitude': 'first',
    'pincode': 'first',
    'day': 'first',
    'month': 'first',
    'quarter': 'first',
    'year_month': 'first',
    'geographical_zone': 'first',
    'Topographical Region': 'first',
    'Geographical_Zone': 'first'
}

df_aggregated = df_main.groupby(['state', 'district', 'date']).agg(agg_funcs).reset_index()

print("Aggregated DataFrame created successfully. Displaying the first 5 rows:")
display(df_aggregated.head())

print("\nShape of original DataFrame (df_main):", df_main.shape)
print("Shape of aggregated DataFrame (df_aggregated):", df_aggregated.shape)


Aggregated DataFrame created successfully. Displaying the first 5 rows:


,state,district,date,age_0_5,age_5_17,age_18_greater,Latitude,Longitude,pincode,day,month,quarter,year_month,geographical_zone,Topographical Region,Geographical_Zone
0,Andhra Pradesh,Anantapur,2025-09-01,76.0,7.0,0.0,14.312066,77.460158,515672.0,1,9,3,2025-09,South,Coastal,West
1,Andhra Pradesh,Anantapur,2025-09-02,17.0,2.0,0.0,14.312066,77.460158,515002.0,2,9,3,2025-09,South,Coastal,West
2,Andhra Pradesh,Chittoor,2025-09-01,121.0,9.0,0.0,13.331093,78.927639,517004.0,1,9,3,2025-09,South,Coastal,South
3,Andhra Pradesh,Chittoor,2025-09-02,43.0,4.0,0.0,13.331093,78.927639,517001.0,2,9,3,2025-09,South,Coastal,South
4,Andhra Pradesh,East Godavari,2025-09-01,125.0,10.0,1.0,16.782718,82.243207,507129.0,1,9,3,2025-09,Central,Coastal,East



Shape of original DataFrame (df_main): (13364, 16)
Shape of aggregated DataFrame (df_aggregated): (1547, 16)


In [ ]:
df_aggregated.to_csv("updated1.csv", index=False)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from google.colab import files

files.download('updated1.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>